In [1]:
import os

In [4]:
# os.chdir("../")

In [3]:
pwd

'/Users/basazinbelhu/Simple_MLOPS'

In [5]:
from dataclasses import dataclass
from pathlib import Path
@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str


In [7]:
from src.simple_mlops.constants import *
from src.simple_mlops.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path=CONFIG_FILE_PATH,
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifact_dir])
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])          # <- ADD THIS

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name,
        )
        return model_trainer_config

In [9]:
import pandas as pd
import os
from src.simple_mlops import logger
from sklearn.linear_model import ElasticNet
import joblib

In [10]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)

        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]

        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state = 45)
        lr.fit(train_x, train_y)
        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))


In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config= model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-06-13 07:59:36,163: INFO: common]: Successfully read YAML file: config/config.yaml
[2026-06-13 07:59:36,165: INFO: common]: Successfully read YAML file: params.yaml
[2026-06-13 07:59:36,167: INFO: common]: Successfully read YAML file: schema.yaml
[2026-06-13 07:59:36,168: INFO: common]: Directory created or already exists: artifact


FileNotFoundError: [Errno 2] No such file or directory: '/artifact/data_transformation/train.csv'